# Loan Default Prediction - Phase 3 Model Analysis

The shared training script fits baseline and tuned models, selects only on validation ROC-AUC, and reads the test split only for final evaluation.

## 1. Load processed training and validation data

In [ ]:
from pathlib import Path
import json
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    ConfusionMatrixDisplay, average_precision_score, precision_recall_curve,
    roc_auc_score, roc_curve,
)

CANDIDATE_ROOTS = [Path.cwd(), Path.cwd().parent, Path.cwd() / "loan-default-prediction", Path.cwd().parent / "loan-default-prediction"]
ROOT = next((path for path in CANDIDATE_ROOTS if (path / "data/processed/train.csv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Could not locate loan-default-prediction/data/processed/train.csv")
sys.path.insert(0, str(ROOT))
train = pd.read_csv(ROOT / "data/processed/train.csv")
validation = pd.read_csv(ROOT / "data/processed/validation.csv")
assert "Default" in train and "Default" in validation
X_train, y_train = train.drop(columns="Default"), train["Default"]
X_validation, y_validation = validation.drop(columns="Default"), validation["Default"]
assert "Default" not in X_train and "Default" not in X_validation
assert X_train.columns.equals(X_validation.columns)
print(f"Train: {train.shape}; Validation: {validation.shape}")

## 2. Train baselines, tune on training rows, and select on validation

In [ ]:
from scripts.train_models import main as run_training

required_outputs = [
    ROOT / "reports/baseline_model_comparison.csv",
    ROOT / "reports/model_comparison.csv",
    ROOT / "models/best_model_metadata.json",
    ROOT / "models/logistic_regression.joblib",
    ROOT / "models/random_forest.joblib",
    ROOT / "models/xgboost.joblib",
]
if not all(path.is_file() for path in required_outputs):
    result = run_training()
    if result != 0:
        raise RuntimeError(f"Training workflow exited with status {result}.")
else:
    print("Existing Phase 3 outputs found; using the saved validation comparison and models.")

## 3. Baseline comparison (validation only)

In [ ]:
baseline = pd.read_csv(ROOT / "reports/baseline_model_comparison.csv")
display(baseline[baseline["Evaluation_Set"] == "Validation"].sort_values("ROC_AUC", ascending=False))

## 4. Tuned model comparison (validation only)

In [ ]:
comparison = pd.read_csv(ROOT / "reports/model_comparison.csv")
tuned_validation = comparison[(comparison["Tuned"] == True) & (comparison["Evaluation_Set"] == "Validation")]
display(tuned_validation.sort_values("ROC_AUC", ascending=False))

## 5. Validation ROC curves

In [ ]:
model_files = {
    "Logistic Regression": "logistic_regression.joblib",
    "Random Forest": "random_forest.joblib",
    "XGBoost": "xgboost.joblib",
}
validation_models = {name: joblib.load(ROOT / "models" / filename) for name, filename in model_files.items()}
plt.figure(figsize=(8, 6))
for name, model in validation_models.items():
    probability = model.predict_proba(X_validation)[:, 1]
    fpr, tpr, _ = roc_curve(y_validation, probability)
    auc = roc_auc_score(y_validation, probability)
    plt.plot(fpr, tpr, label=f"{name} (ROC-AUC={auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="grey", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Validation ROC curves")
plt.legend()
plt.tight_layout()
plt.show()

## 6. Validation precision-recall curves

In [ ]:
plt.figure(figsize=(8, 6))
for name, model in validation_models.items():
    probability = model.predict_proba(X_validation)[:, 1]
    precision, recall, _ = precision_recall_curve(y_validation, probability)
    average_precision = average_precision_score(y_validation, probability)
    plt.plot(recall, precision, label=f"{name} (AP={average_precision:.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Validation precision-recall curves")
plt.legend()
plt.tight_layout()
plt.show()

## 7. Validation confusion matrices at threshold 0.50

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for axis, (name, model) in zip(axes, validation_models.items()):
    predictions = (model.predict_proba(X_validation)[:, 1] >= 0.50).astype(int)
    ConfusionMatrixDisplay.from_predictions(
        y_validation, predictions, labels=[0, 1],
        display_labels=["No Default", "Default"], cmap="Blues", colorbar=False, ax=axis,
    )
    axis.set_title(name)
fig.suptitle("Validation confusion matrices (threshold=0.50)")
fig.tight_layout()
plt.show()

## 8. Selected model and final test evaluation

In [ ]:
metadata = json.loads((ROOT / "models/best_model_metadata.json").read_text(encoding="utf-8"))
final_test = comparison[comparison["Evaluation_Set"] == "Test (final)"]
selected_model_name = metadata["model_name"]
validation_roc_auc = metadata["validation_roc_auc"]
test_roc_auc = metadata["test_roc_auc"]
print(f"Selected by validation ROC-AUC: {selected_model_name}")
print(f"Validation ROC-AUC: {validation_roc_auc:.4f}")
print("Final test metrics (test data was not used for tuning or selection):")
display(final_test)
target_status = "achieved" if test_roc_auc >= 0.85 else "not achieved"
print(f"Target >= 0.85: {target_status}")